# 01 — Primary Simulation

Monte Carlo simulation of 1,000 clinical AI tools under the heterogeneous evidence model.
Produces primary metrics, bootstrap CIs, Epic case scenario, and figures 1–4.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

OUT_FIGURES = REPO_ROOT / "outputs" / "figures"
OUT_TABLES  = REPO_ROOT / "outputs" / "tables"
OUT_DATA    = REPO_ROOT / "outputs" / "data"
OUT_LOGS    = REPO_ROOT / "outputs" / "logs"
for d in [OUT_FIGURES, OUT_TABLES, OUT_DATA, OUT_LOGS]:
    d.mkdir(parents=True, exist_ok=True)

import json, numpy as np, pandas as pd
from run_simulation import (
    SimConfig, load_config, simulate_portfolio,
    decide_noncomp_gates, decide_weighted_composite, decide_permissive,
    compute_metrics, bootstrap_ci, set_threshold_to_match_rate,
    epic_case_row, fig1_unsafe_rates, fig2_tradeoff_curve,
    fig3_gate_failures, fig4_epic_case,
)

In [2]:
cfg = load_config(str(REPO_ROOT / "src" / "params_default.json"))
print(f"n_tools={cfg.n_tools}, seed={cfg.seed}, p_high_risk={cfg.p_high_risk}")

n_tools=1000, seed=20260304, p_high_risk=0.3


In [3]:
df = simulate_portfolio(cfg)
print(f"Portfolio: {len(df)} tools, {int(df['unsafe_true'].sum())} unsafe, {int((df['risk_tier']=='high').sum())} high-risk")

Portfolio: 1000 tools, 196 unsafe, 326 high-risk


In [4]:
# Apply all decision rules
df = decide_noncomp_gates(df, cfg)
weights = cfg.composite_weights
gate_deploy_rate = float(df["deploy_gate"].mean())

tmp = decide_weighted_composite(df, weights, threshold=0.0, missing_mode="mean")
scores_mean = tmp["composite_score_mean"].to_numpy()
thr_matched = set_threshold_to_match_rate(scores_mean, gate_deploy_rate)
thr_moderate = set_threshold_to_match_rate(scores_mean, min(gate_deploy_rate * 2.2, 0.85))

df = decide_weighted_composite(df, weights, threshold=thr_matched, missing_mode="mean", col_suffix="matched")
df = decide_weighted_composite(df, weights, threshold=thr_moderate, missing_mode="mean")

tmp = decide_weighted_composite(df, weights, threshold=0.0, missing_mode="zero", col_suffix="zero_tmp")
scores_zero = tmp["composite_score_zero_tmp"].to_numpy()
thr_zero_mod = set_threshold_to_match_rate(scores_zero, min(gate_deploy_rate * 2.2, 0.85))
df = decide_weighted_composite(df, weights, threshold=thr_zero_mod, missing_mode="zero")

df = decide_permissive(df, cfg)
print(f"Gate deployment rate: {gate_deploy_rate:.3f}")

Gate deployment rate: 0.285


In [5]:
# Compute metrics and bootstrap CIs
method_cols = {
    "Gates": "deploy_gate", "Composite (matched)": "deploy_composite_matched",
    "Composite (mean)": "deploy_composite_mean", "Composite (zero)": "deploy_composite_zero",
    "Permissive": "deploy_permissive",
}
metrics = {}
for label, col in method_cols.items():
    metrics[label] = compute_metrics(df, col)
metrics["Gates"]["override_rate"] = float(df["override_used"].mean())
metrics["Gates"]["n_overrides"] = int(df["override_used"].sum())

ci_data = {}
for label, col in method_cols.items():
    ci_data[label] = {}
    for mk in ["unsafe_deployment_rate", "unsafe_among_deployed"]:
        ci_data[label][mk] = bootstrap_ci(df, col, mk, n_boot=cfg.n_bootstrap, seed=cfg.seed + 42)

for label in ["Gates", "Composite (mean)", "Permissive"]:
    m = metrics[label]
    print(f"{label:25s}  Deploy={m['deployment_rate']:.3f}  Unsafe={m['unsafe_deployment_rate']:.4f}")

Gates                      Deploy=0.285  Unsafe=0.0000
Composite (mean)           Deploy=0.627  Unsafe=0.0090
Permissive                 Deploy=0.760  Unsafe=0.0220


In [6]:
# Save primary data
df.to_csv(OUT_DATA / "simulation_outputs.csv", index=False, lineterminator="\n")
summary = {
    "config": {"n_tools": cfg.n_tools, "seed": cfg.seed, "p_high_risk": cfg.p_high_risk,
        "p_unsafe_standard": cfg.p_unsafe_standard, "p_unsafe_high": cfg.p_unsafe_high,
        "obs_noise_sd": cfg.obs_noise_sd, "maturity_effect_sd": cfg.maturity_effect_sd,
        "override": {"allow_override": cfg.allow_override, "override_prob": cfg.override_prob,
            "override_max_failed_gates": cfg.override_max_failed_gates,
            "override_disallowed_gates": list(cfg.override_disallowed_gates)},
        "thresholds_standard": cfg.thresholds_standard, "thresholds_high": cfg.thresholds_high,
        "weights": weights,
        "composite_thresholds": {"matched_to_gates": thr_matched, "moderate_mean": thr_moderate,
            "moderate_zero": thr_zero_mod, "gate_deploy_rate": gate_deploy_rate}},
    "metrics": {k: v for k, v in metrics.items()},
    "bootstrap_ci": {k: {mk: list(v) for mk, v in ci.items()} for k, ci in ci_data.items()},
}
(OUT_DATA / "metrics_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8", newline="\n")
print("Saved simulation_outputs.csv and metrics_summary.json")

Saved simulation_outputs.csv and metrics_summary.json


In [7]:
# Generate figures 1-3
fig1_unsafe_rates(metrics, OUT_FIGURES)
fig2_tradeoff_curve(df, OUT_FIGURES)
fig3_gate_failures(df, cfg, OUT_FIGURES)
print("Generated figures 1-3")

Generated figures 1-3


In [8]:
# Epic case scenario
case = pd.DataFrame([epic_case_row()])
case = decide_noncomp_gates(case, cfg)
case = decide_weighted_composite(case, weights, threshold=thr_matched, missing_mode="mean", col_suffix="matched")
case = decide_weighted_composite(case, weights, threshold=thr_moderate, missing_mode="mean")
case = decide_weighted_composite(case, weights, threshold=thr_zero_mod, missing_mode="zero")
case.to_csv(OUT_DATA / "epic_case_outputs.csv", index=False, lineterminator="\n")
fig4_epic_case(case, cfg, OUT_FIGURES)
print(f"Epic case: {case.iloc[0]['n_failed_gates']}/5 gates failed, deploy={case.iloc[0]['deploy_gate']}")

C:\Users\Walter.Brown\EAA_Portfolio\ethical-alpha-audit-paper-2-threshold-justification\src\run_simulation.py:304: RuntimeWarning: Mean of empty slice
  col_means = np.nanmean(obs_mat, axis=0)


Epic case: 5/5 gates failed, deploy=0


## Headline Metrics

| Method | Deploy Rate | Unsafe Rate | 95% CI |
|---|---|---|---|
| Gates | 28.5% | 0.0% | [0.0%, 0.0%] |
| Composite (moderate) | 62.7% | 0.9% | [0.4%, 1.6%] |
| Permissive | 76.0% | 2.2% | [1.3%, 3.2%] |